# 팀원 GPU: 세로 어텐션 4개 모델 / 1000epoch
위에서 아래로 실행한다. 이 노트북 파일 자체는 학습을 실행하지 않은 상태로 배포한다.

| 담당 | 조건 | head | 개수 |
|---|---|---|---:|
| 팀원 | 소형2D 전처리 + 세로 어텐션 + local CNN | Pooled, Self | 2 |
| 팀원 | 세로 어텐션 + 세로 CNN(local 경로 포함) | Pooled, Self | 2 |
| 기존 서버 | 세로 어텐션 + 기존 local CNN | Mean, Pooled, Self | 3 |
| 기존 서버 | 위의 소형2D / 세로CNN 조건 | 각각 Mean | 2 |

현재 기존 서버에서 진행 중인 세로 어텐션 **없는** 6개는 별도다. 기존 서버 담당 어텐션5개는 아직 시작하지 않았다.
기존 `experiments.frequency_local.launch`는 전체15개를 실행하므로 **호출하지 않는다**.

동일 조건: 화자03 개발검증203개, 화자02/04/05 학습609개, seed42, 사전학습/검출기 없음,
전체1.5초·64mel, 동일 포먼트 캐시와 소음, CE+0.1SupCon(same_command), temperature0.1,
AdamW1e-3→1e-5 cosine1000, batch32 녹음(64paired views).
세로 어텐션은 CNN 앞(소형2D 조건에서는 전처리 뒤), 주파수64개 유지, d16/2heads/FF32/SDPA.
선택 checkpoint는 0dB draw0 최고점, 최종 평가는8소음×3draw. 독립 test01은 읽지 않는다.


## 0. 커널 준비
Linux NVIDIA GPU + Python3.10 또는3.11 권장. 아래 설치 셀은 전용 환경에서 한 번 실행하고,
이미 torch를 import했다면 커널을 재시작한 뒤 1번부터 실행한다.
루트 requirements.txt는 다른 버전이므로 이번 실행에 그대로 사용하지 않는다.
호스트 NVIDIA 드라이버가 CUDA12.1 런타임과 호환되어야 한다.
환경이 이미 동일하면 설치 셀을 건너뛴다. 원격 GPU의 영구 저장 공간을 사용한다.


In [ ]:
# 필요한 경우에만 주석을 해제해 설치한다. 설치 후 커널 재시작.
# %pip install torch==2.3.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
# %pip install numpy==1.24.4 pandas==2.2.2 scipy==1.14.0 soundfile==0.14.0 scikit-learn==1.5.1 praat-parselmouth==0.4.7


## 1. 경로·고정 소스 체크아웃
WORK는 충분한 여유 공간이 있는 영구 경로로 변경한다. 이 커밋에 모델별 분리 실행 초기값 보정 코드가 포함된다.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib, shutil, urllib.request, tarfile, time
WORK = Path.home() / 'audio-team4'
# 이미 전용 브랜치를 clone했다면 그 저장소 경로로 변경한다.
REPO = WORK / 'CAS2250-audio-repo'
BRANCH = 'experiment/frequency-attention-team4'
SOURCE_REFERENCE = '04aaae6cab99563ace0aec2fd050d9a5374cfe99'
URL = 'https://github.com/CAS2250-audio-task/CAS2250-audio-repo.git'
WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,URL,str(REPO)],check=True)
subprocess.run(['git','fetch','origin',BRANCH],cwd=REPO,check=True)
actual_branch = subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()
assert actual_branch == BRANCH, f'전용 브랜치 필요: 현재 {actual_branch}'
# 기준 학습 코드와 같은지 확인. 문서/노트북 변경은 허용한다.
subprocess.run(['git','diff','--exit-code',SOURCE_REFERENCE,'HEAD','--','experiments'],cwd=REPO,check=True)
subprocess.run(['git','diff','--exit-code','HEAD','--','experiments'],cwd=REPO,check=True)
SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('SOURCE',SOURCE_COMMIT,'BRANCH',BRANCH,'REPO',REPO)


## 2. GPU·환경 확인
다른 GPU/라이브러리에서는 완전한 bitwise 동일 결과는 보장하지 않는다. 환경 기록을 결과와 함께 전달한다.


In [ ]:
import torch, torchaudio, numpy, pandas, soundfile, importlib.metadata as meta
assert torch.cuda.is_available(), 'GPU 커널/드라이버 확인'
assert torch.__version__ == '2.3.1+cu121', torch.__version__
assert torchaudio.__version__ == '2.3.1+cu121', torchaudio.__version__
assert numpy.__version__ == '1.24.4'
ENV = dict(python=sys.version, gpu=torch.cuda.get_device_name(0),
           packages={p:meta.version(p) for p in ['torch','torchaudio','numpy','pandas','scipy','soundfile','scikit-learn','praat-parselmouth']})
(WORK/'environment.json').write_text(json.dumps(ENV,indent=2))
print(json.dumps(ENV,indent=2))
print('free disk GiB:', round(shutil.disk_usage(WORK).free/2**30,1))


## 3. 원본·증강 캐시 복원
공개 릴리스 약1GB를 한 번 다운로드한다. 원본812개, 포먼트 캐시, 준비된 소음과 manifest를 포함한다.
기존 cache를 임의로 다시 만들거나 자기 데이터로 교체하지 않는다. GitHub CLI 인증 없이 다운로드 가능.
복원은 기존 실험 파일을 덮을 수 있으므로 위의 전용 checkout을 사용한다.


In [ ]:
ASSET = WORK/'compact-pooling-v1-assets.tar.gz'
ASSET_SHA256 = 'fb7ed56c69fa036053be5cf9572853f119c3226fa78422d4b50a899e2de78e10'
ASSET_URL = 'https://github.com/CAS2250-audio-task/CAS2250-audio-repo/releases/download/compact-pooling-v1-assets/compact-pooling-v1-assets.tar.gz'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
if not ASSET.exists():
    partial=ASSET.with_suffix('.partial')
    request=urllib.request.Request(ASSET_URL,headers={'User-Agent':'audio-team4'})
    with urllib.request.urlopen(request,timeout=120) as source, partial.open('wb') as target:
        shutil.copyfileobj(source,target,length=1<<20)
    partial.replace(ASSET)
assert sha(ASSET)==ASSET_SHA256, '다운로드 무결성 불일치: 실행 중단'
MARKER=WORK/'assets-restored.json'
if not MARKER.exists():
    subprocess.run([sys.executable,'-m','experiments.compact_pooling.bundle','restore',str(ASSET)],cwd=REPO,check=True)
    MARKER.write_text(json.dumps(dict(archive_sha256=ASSET_SHA256,repo=str(REPO))))
else:
    assert json.loads(MARKER.read_text())==dict(archive_sha256=ASSET_SHA256,repo=str(REPO))
manifest=json.loads((REPO/'runs/compact-pooling-v1/formants.json').read_text())
assert sha(REPO/'runs/compact-pooling-v1/formants.npy')==manifest['sha256']
for key,path in [('transform_source_sha256','experiments/overnight/data.py'),('settings_source_sha256','experiments/overnight/waveforms.py')]:
    assert sha(REPO/path)==manifest['identity'][key]
print('ASSETS VERIFIED')


## 4. 두 그룹·4개 모델 실행
두 그룹을 **순차 실행**한다. 각 그룹 안에서 Pooled/Self만 학습한다. 새 split 진입점은 분류기와
SupCon projector 초기값을 전체3종 실행과 동일하게 보존한다.
노트북 셀은 감독 프로세스만 시작하고 즉시 반환한다. 브라우저를 닫아도 서버 프로세스가 유지되는 환경이면 계속 돈다.
클라우드 workspace 자체 종료/유휴 정지 정책은 별개다.

10epoch마다 optimizer 포함 last.pt 저장. 중단 후 같은 코드·경로·옵션으로 재실행하면 최근 checkpoint에서 재개한다.
학습 중 소스 수정/git pull/다른 버전 설치 금지. 이미 진행 중이면 이 셀을 중복 실행하지 않는다.


In [ ]:
JOBS=[]
for cnn in ['lowcost','frequency']:
    name=f'frequency-local-{cnn}-v1-teamPS-spk03-e1000'
    command=[sys.executable,'-u','-m','experiments.frequency_local_subset',
             '--cnn',cnn,'--vertical','--models','cnn_pooled_query','cnn_self_cls',
             '--objective','paired_supcon','--positive-mode','same_command',
             '--supcon-weight','0.1','--temperature','0.1','--folds','spk03',
             '--epochs','1000','--output',str(REPO/'runs'/name)]
    JOBS.append(dict(cnn=cnn,name=name,command=command))
CONTROL=WORK/'team4-control';CONTROL.mkdir(exist_ok=True)
CONFIG=CONTROL/'config.json'
CONFIG.write_text(json.dumps(dict(repo=str(REPO),jobs=JOBS),indent=2))
SUPERVISOR=CONTROL/'supervisor.py'
SUPERVISOR.write_text(r"""
import json,subprocess,sys,datetime,os
from pathlib import Path
control=Path(__file__).resolve().parent
config=json.loads((control/'config.json').read_text())
state=dict(status='running',pid=os.getpid(),groups={})
def save():
    temp=control/'status.tmp';temp.write_text(json.dumps(state,indent=2));temp.replace(control/'status.json')
save()
try:
    for job in config['jobs']:
        with (control/(job['cnn']+'.log')).open('a',buffering=1) as log:
            child=subprocess.Popen(job['command'],cwd=config['repo'],stdout=log,stderr=subprocess.STDOUT)
            state['groups'][job['cnn']]=dict(status='running',pid=child.pid,name=job['name'])
            save();code=child.wait()
        state['groups'][job['cnn']].update(status='complete' if code==0 else 'failed',returncode=code);save()
        if code:raise RuntimeError(job['cnn']+' failed: inspect log')
    state['status']='complete';save()
except Exception as error:
    state.update(status='failed',error=str(error));save();raise
""")
previous=CONTROL/'pid.json'
if previous.exists():
    pid=json.loads(previous.read_text())['pid']
    try: os.kill(pid,0)
    except ProcessLookupError: pass
    else: raise RuntimeError(f'PID {pid} 있음: 중복 실행하지 말고 상태 먼저 확인')
with (CONTROL/'supervisor.log').open('a') as log:
    supervisor=subprocess.Popen([sys.executable,'-u',str(SUPERVISOR)],cwd=REPO,
        stdout=log,stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True)
previous.write_text(json.dumps(dict(pid=supervisor.pid)))
print('STARTED',supervisor.pid,'/ logs:',CONTROL)


## 5. 진행·ETA 확인
아래 셀을 필요할 때 실행한다. JSON만 읽어 GPU 연산을 추가하지 않는다. 자동 갱신하려면 `while` 루프에 30초 sleep을 넣어도 된다. ETA는 현재 그룹 기준이며 미시작 그룹 시간은 별도다.


In [ ]:
state=json.loads((CONTROL/'status.json').read_text())
print('SUPERVISOR:',state['status'])
for job in JOBS:
    status=state.get('groups',{}).get(job['cnn'],{}).get('status','queued')
    path=REPO/'runs'/job['name']/'progress.json'
    progress=json.loads(path.read_text()) if path.exists() else {}
    print(job['cnn'],status,'epoch',1000 if status=='complete' else progress.get('epoch','-'),
          '/1000; 현재 그룹 ETA(분):',progress.get('estimated_training_minutes_remaining','-'))
# 상세 로그: CONTROL/'lowcost.log', CONTROL/'frequency.log'


## 6. 완료 결과 확인·패키징
두 그룹 완료 후만 실행. 각 summary가2개 모델·각 history가2000행인지 확인한다.
best.pt(선택 가중치)와 last.pt(1000epoch 종료점)는 다르므로 둘 다 보존한다.
원본·소음·증강 캐시는 다시 보낼 필요 없이 릴리스 hash를 참조한다.
생성된 tar.gz와 SHA256 파일을 기존 서버 담당자에게 전달한다. 자동 업로드는 하지 않는다.


In [ ]:
state=json.loads((CONTROL/'status.json').read_text());assert state['status']=='complete',state
rows=[];paths=[]
for job in JOBS:
    run=REPO/'runs'/job['name'];report=REPO/'reports'/job['name']
    summary=json.loads((report/'summary.json').read_text());assert len(summary)==2
    history=json.loads((run/'spk03/history.json').read_text());assert len(history)==2000
    assert {x['model'] for x in summary}=={'cnn_pooled_query','cnn_self_cls'}
    for name in ['cnn_pooled_query','cnn_self_cls']:
        assert max(x['epoch'] for x in history if x['model']==name)==1000
    for row in summary:
        rows.append(dict(cnn=job['cnn'],model=row['model'],epoch=row['best_epoch'],
                         original_f1=row['best']['original_f1'],zero_f1=row['best']['zero_f1']))
    paths += [p for folder in [run,report] for p in folder.rglob('*') if p.is_file()]
manifest=dict(source_commit=SOURCE_COMMIT,asset_sha256=ASSET_SHA256,environment=ENV,summary=rows,
              files=[dict(path=str(p.relative_to(REPO)),sha256=sha(p),bytes=p.stat().st_size) for p in paths])
MANIFEST=WORK/'team4-manifest.json';MANIFEST.write_text(json.dumps(manifest,indent=2))
ARCHIVE=WORK/'frequency-attention-team4-results.tar.gz'
with tarfile.open(ARCHIVE,'w:gz') as tar:
    tar.add(MANIFEST,arcname='team4-manifest.json')
    tar.add(WORK/'environment.json',arcname='team4-environment.json')
    for p in paths:tar.add(p,arcname=str(p.relative_to(REPO)),recursive=False)
ARCHIVE.with_suffix(ARCHIVE.suffix+'.sha256').write_text(f'{sha(ARCHIVE)}  {ARCHIVE.name}\n')
print(pandas.DataFrame(rows).to_string(index=False))
print('전달:',ARCHIVE,ARCHIVE.with_suffix(ARCHIVE.suffix+'.sha256'))


## 중단이 필요할 때만
Jupyter의 셀 중지는 백그라운드 학습을 멈추지 않는다. 중단하려면 아래 코드를 별도 셀에서 **명시적으로** 실행한다.
감독 프로세스 그룹과 두 모델 학습을 함께 종료한다. 이 GPU의 다른 작업은 건드리지 않는다.
```python
import signal
pid = json.loads((CONTROL/'pid.json').read_text())['pid']
assert os.getpgid(pid) == pid
os.killpg(pid, signal.SIGTERM)
```
재개할 때는 소스·설정을 유지하고 실행 셀을 다시 실행한다. 실패하면 해당 그룹 log 끝부분을 먼저 전달한다.


## 7. 결과를 전용 원격 브랜치와 릴리스에 업로드
먼저 Jupyter 서버 터미널에서 GitHub CLI를 설치하고 **본인 계정**으로 `gh auth login`,
`gh auth setup-git`을 실행한다. 저장소 쓰기 권한이 필요하며, 없으면 관리자에게 collaborator 권한을 요청한다.
토큰을 노트북/채팅/저장소에 붙여넣지 않는다. Git author가 없으면 전용 checkout에서 본인 이름·이메일로
`git config user.name "본인 이름"`, `git config user.email "본인 이메일"`을 설정한다.

아래 셀은 실제 업로드를 수행한다. **두 그룹 완료·6번 패키징 이후** 실행한다.
Git에는 두 그룹의 reports만 올리고, 가중치/optimizer/epoch기록/증강trace는 결과 tar.gz를 릴리스에 올린다.
원본 음원·캐시는 기존 릴리스를 재사용하므로 다시 업로드하지 않는다. 강제 push는 사용하지 않는다.
릴리스 태그가 이미 있으면 중단한다. 재실험이면 담당자와 상의해 v2 등 새 태그를 사용한다.


In [ ]:
assert json.loads((CONTROL/'status.json').read_text())['status']=='complete'
assert ARCHIVE.exists() and ARCHIVE.with_suffix(ARCHIVE.suffix+'.sha256').exists()
assert subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()==BRANCH
subprocess.run(['gh','auth','status'],cwd=REPO,check=True)
paths=[str(Path('reports')/job['name']) for job in JOBS]
subprocess.run(['git','add','--']+paths,cwd=REPO,check=True)
# 전용 checkout에서 다른 변경을 stage하지 않는다
staged=subprocess.check_output(['git','diff','--cached','--name-only'],cwd=REPO,text=True).splitlines()
assert all(any(path.startswith(prefix+'/') for prefix in paths) for path in staged), staged
if staged:
    subprocess.run(['git','commit','-m','Complete team4 vertical attention experiments: pooled and self, 1000 epochs'],cwd=REPO,check=True)
# 원격 변경을 반영하고 push한다. 충돌하면 중단하고 담당자에게 알린다.
subprocess.run(['git','pull','--rebase','origin',BRANCH],cwd=REPO,check=True)
subprocess.run(['git','push','-u','origin',BRANCH],cwd=REPO,check=True)
RESULT_TAG='frequency-attention-team4-results-v1'
result_repo='CAS2250-audio-task/CAS2250-audio-repo'
exists=subprocess.run(['gh','release','view',RESULT_TAG,'--repo',result_repo],capture_output=True)
assert exists.returncode != 0, '기존 릴리스를 덮어쓰지 말고 새 태그를 사용'
notes=CONTROL/'release-notes.md'
notes.write_text('Team4: lowcost/frequency CNN + vertical attention, Pooled/Self, speaker03, 1000 epochs.\n'
                 f'Training source: {SOURCE_COMMIT}\nInput asset SHA256: {ASSET_SHA256}\n'
                 'Contains selected best weights, last checkpoints, histories, traces and predictions.\n'
                 'Development validation; not independent test results. See team4-manifest.json.\n')
subprocess.run(['gh','release','create',RESULT_TAG,str(ARCHIVE),str(ARCHIVE.with_suffix(ARCHIVE.suffix+'.sha256')),
                '--repo',result_repo,'--target',BRANCH,
                '--title','Team4 frequency attention: four completed models',
                '--notes-file',str(notes)],cwd=REPO,check=True)
print('보고서:',f'https://github.com/{result_repo}/tree/{BRANCH}')
print('가중치:',f'https://github.com/{result_repo}/releases/tag/{RESULT_TAG}')
